# 51. 서울코퍼스 발음 검색 (전체 음운현상)

30번(34-36) + 37번 사전검색 결과의 단어가 서울 코퍼스에서 실제로 어떻게 발음되는지 확인

## 검색 대상 음운현상
- **34** ㄴ/ㄹ삽입
- **35** 유음화/비음화
- **36** 합성어경음화
- **37** 모음조화/충돌

## 입력
- 30번 결과 CSV (34-36)
- 37번 결과 CSV
- 서울 enriched: `04_seoul_pword_enriched.csv` (242MB)

## 출력
- 전체 매칭 CSV: `search_results/seoul_all_phenomena_*.csv`
- 구 경계 ㄴ삽입 CSV: `search_results/seoul_phrase_n_insertion_*.csv`

## 1. 환경 설정

In [7]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
except ImportError:
    PROJECT_ROOT = os.path.dirname(os.getcwd())
    print(f'Local mode: {PROJECT_ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re

# utils_phonology.py 로드
UTILS_PATH = f'{PROJECT_ROOT}/30_search_dictionary'
os.chdir(UTILS_PATH)
%run utils_phonology.py

# 경로 설정
SEARCH_RESULTS_30 = f'{PROJECT_ROOT}/30_search_dictionary/search_results'
SEARCH_RESULTS_37 = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
SEOUL_PWORD = f'{PROJECT_ROOT}/00_raw_data/03_seoul_corpus/02_csv/04_seoul_pword_enriched.csv'
RESULT_DIR = f'{PROJECT_ROOT}/50_search_seoul_corpus/search_results'
os.makedirs(RESULT_DIR, exist_ok=True)

[OK] utils_phonology.py 로드 완료
   함수 34개


## 2. 데이터 로드

In [9]:
# 34-37번 결과에서 단어 목록 + 현상 태그 추출
PHENOMENON_MAP = {
    'n_l_insertion': '34_n_l_insertion',
    'nl_ln_nasalization': '35_nasalization',
    'fortis_compound': '36_fortis',
    'vowel_harmony_collision': '37_vowel',
    'vowel_collision': '37_vowel',
}

def classify_phenomenon(filename):
    for key, label in PHENOMENON_MAP.items():
        if key in filename:
            return label
    return 'unknown'

# 30번 결과
result_files_30 = list(Path(SEARCH_RESULTS_30).glob('*.csv'))
# 37번 결과 (사전검색 원본만, ls_freq/seoul/dialogue 제외)
result_files_37 = [f for f in Path(SEARCH_RESULTS_37).glob('*.csv')
                   if not f.name.startswith(('ls_freq_', 'seoul_', 'dialogue_'))]

# 단어 → 현상 매핑 (한 단어가 여러 현상에 해당할 수 있음)
word_to_phenomena = {}
for f in result_files_30 + result_files_37:
    phenom = classify_phenomenon(f.stem)
    df_tmp = pd.read_csv(f, encoding='utf-8-sig', usecols=['word'])
    for w in df_tmp['word'].unique():
        if w not in word_to_phenomena:
            word_to_phenomena[w] = set()
        word_to_phenomena[w].add(phenom)

all_words = set(word_to_phenomena.keys())
print(f'전체 고유 단어: {len(all_words):,}개')
for phenom in sorted(set(p for ps in word_to_phenomena.values() for p in ps)):
    count = sum(1 for ps in word_to_phenomena.values() if phenom in ps)
    print(f'  {phenom}: {count:,}개')

전체 고유 단어: 204,656개
  34_n_l_insertion: 21,818개
  35_nasalization: 18,025개
  36_fortis: 118,354개
  37_vowel: 59,597개
  unknown: 17개


In [10]:
# 서울 pWord enriched 로드 (242MB)
print('서울 pWord 로딩...')
df_seoul = pd.read_csv(SEOUL_PWORD, encoding='utf-8-sig', low_memory=False)
print(f'서울 pWord: {len(df_seoul):,}행')
print(f'컬럼: {list(df_seoul.columns)}')
df_seoul.head(3)

서울 pWord 로딩...
서울 pWord: 231,632행
컬럼: ['file_no', 'filename', 'speaker_id', 'spk_gender', 'spk_age', 'interviewer_gender', 'word_no', 'word_xmin', 'word_xmax', 'word_duration', 'pWord_ortho', 'pWord_ortho_seoul', 'pWord_prono', 'pWord_prono_seoul', 'utt_ortho', 'utt_prono', 'utt_xmin', 'utt_xmax', 'pWord_ortho_roman', 'pWord_prono_roman', 'morphs', 'morphs_roman', 'morphs_v7_ids', 'morphs_v7_origins']


,file_no,filename,speaker_id,spk_gender,spk_age,interviewer_gender,word_no,word_xmin,word_xmax,word_duration,...,utt_ortho,utt_prono,utt_xmin,utt_xmax,pWord_ortho_roman,pWord_prono_roman,morphs,morphs_roman,morphs_v7_ids,morphs_v7_origins
0,1,s01m16f1,1,m,16,f,1,0.937370,1.193120,0.255749,...,네,네,0.937370,1.193120,N E,N E,네/IC,N E/IC,"네:179477:003,179477:004,179477:005,179477:006,...",NaN
1,1,s01m16f1,1,m,16,f,2,8.342122,8.500322,0.158201,...,제 이름은,제 이르믄,8.342122,8.994354,J E,J E,저/NP+의/JKG 이름/NNG+은/JX,J EO/NP+euI/JKG I R EU M/NNG+EU N/JX,"저:796829:003,796829:004,796830:005,796830:006|...",nan|nan
2,1,s01m16f1,1,m,16,f,3,8.500322,8.994354,0.494031,...,제 이름은,제 이르믄,8.342122,8.994354,I L EU M EU N,I L EU M EU N,저/NP+의/JKG 이름/NNG+은/JX,J EO/NP+euI/JKG I R EU M/NNG+EU N/JX,"저:796829:003,796829:004,796830:005,796830:006|...",nan|nan


## 3. 단어 매칭 (전체 음운현상)

In [11]:
# pWord_ortho 기준으로 전체 단어 매칭
df_matched = df_seoul[df_seoul['pWord_ortho'].isin(all_words)].copy()

# 현상 태그 추가
df_matched['phenomenon'] = df_matched['pWord_ortho'].map(
    lambda w: ','.join(sorted(word_to_phenomena.get(w, set())))
)

# 철자 vs 발음 차이
df_matched['ortho_prono_diff'] = df_matched['pWord_ortho_roman'] != df_matched['pWord_prono_roman']

# include / realized 컬럼
df_matched['include'] = ''
df_matched['realized'] = ''

n_diff = df_matched['ortho_prono_diff'].sum()
print(f'매칭된 pWord: {len(df_matched):,}행')
print(f'매칭된 고유 단어: {df_matched["pWord_ortho"].nunique():,}개')
print(f'철자!=발음: {n_diff:,}행 ({n_diff/len(df_matched)*100:.1f}%)')
print(f'\n현상별 매칭:')
for phenom in sorted(set(p for ps in word_to_phenomena.values() for p in ps)):
    mask = df_matched['phenomenon'].str.contains(phenom, na=False)
    print(f'  {phenom}: {mask.sum():,}행')

매칭된 pWord: 4,918행
매칭된 고유 단어: 1,352개
철자!=발음: 2,844행 (57.8%)

현상별 매칭:
  34_n_l_insertion: 1,069행
  35_nasalization: 728행
  36_fortis: 2,527행
  37_vowel: 729행
  unknown: 0행


In [12]:
# 사전에 없는 단어 확인 (코퍼스에만 있는 경우)
seoul_words = set(df_seoul['pWord_ortho'].unique())
only_in_corpus = seoul_words - all_words
only_in_dict = all_words - seoul_words
print(f'코퍼스에만 있는 단어: {len(only_in_corpus):,}')
print(f'사전에만 있는 단어: {len(only_in_dict):,}')
print(f'매칭률: {len(all_words & seoul_words) / len(all_words) * 100:.1f}%')

코퍼스에만 있는 단어: 36,398
사전에만 있는 단어: 203,304
매칭률: 0.7%


## 4. 화자 변수별 발음 변이

In [13]:
# 성별/연령별 발음 변이율
if 'spk_gender' in df_matched.columns:
    gender_stats = df_matched.groupby('spk_gender')['ortho_prono_diff'].agg(['sum', 'count', 'mean'])
    gender_stats.columns = ['변이_건수', '전체_건수', '변이율']
    print('성별별 발음 변이:')
    print(gender_stats)

if 'spk_age' in df_matched.columns:
    age_stats = df_matched.groupby('spk_age')['ortho_prono_diff'].agg(['sum', 'count', 'mean'])
    age_stats.columns = ['변이_건수', '전체_건수', '변이율']
    print('\n연령별 발음 변이:')
    print(age_stats)

성별별 발음 변이:
            변이_건수  전체_건수       변이율
spk_gender                        
f            1226   2126  0.576670
m            1618   2792  0.579513

연령별 발음 변이:
         변이_건수  전체_건수       변이율
spk_age                        
15         173    312  0.554487
16         292    503  0.580517
17          93    160  0.581250
18          46     94  0.489362
22         112    195  0.574359
23         209    346  0.604046
24         208    342  0.608187
25          58    101  0.574257
26          63    117  0.538462
27          58    115  0.504348
31         129    218  0.591743
32         200    389  0.514139
34          42     80  0.525000
36         151    269  0.561338
37         176    308  0.571429
38          45    101  0.445545
43         484    782  0.618926
44          86    146  0.589041
46         143    230  0.621739
47          76    110  0.690909


## 5. 구 경계 ㄴ삽입 검색 (핵심)

연속 어절에서 어절① 종성(자음) + 어절② 초성(i/j) → ㄴ삽입 환경

사전(30번)에서는 형태소 경계/단어내부만 검색 가능.
코퍼스에서만 구 경계 ㄴ삽입을 탐지할 수 있다.

In [14]:
def search_phrase_boundary_n_insertion(df, file_col='filename', word_no_col='word_no'):
    """
    연속 어절(pWord)에서 구 경계 ㄴ삽입 환경 탐지

    조건: 어절① 끝 자음 + 어절② 시작 i/j(y)
    """
    candidates = []

    # 파일별로 그룹화 (동일 발화 내 연속 어절만 검색)
    for fname, group in df.groupby(file_col):
        group_sorted = group.sort_values(word_no_col)
        rows = group_sorted.to_dict('records')

        for i in range(len(rows) - 1):
            w1 = rows[i]
            w2 = rows[i + 1]

            # 어절 번호가 연속인지 확인
            if w2.get(word_no_col, 0) != w1.get(word_no_col, 0) + 1:
                continue

            r1 = str(w1.get('pWord_ortho_roman', ''))
            r2 = str(w2.get('pWord_ortho_roman', ''))

            if not r1 or not r2 or r1 == 'nan' or r2 == 'nan':
                continue

            # 어절① 마지막 음절
            syl1_last = r1.split('-')[-1]
            # 어절② 첫 음절
            syl2_first = r2.split('-')[0]

            if ends_with_consonant_roman(syl1_last) and starts_with_i_j_roman(syl2_first):
                # 발음에서 실제 ㄴ삽입 확인
                p2 = str(w2.get('pWord_prono_roman', ''))
                p2_first = p2.split('-')[0] if p2 and p2 != 'nan' else ''
                actual_n = 'yes' if p2_first.upper().startswith('N') else 'no'

                c1, _ = ends_with_consonant_type(syl1_last)
                ins_type = 'ㄹ삽입' if (c1 and c1.lower() == 'l') else 'ㄴ삽입'

                candidates.append({
                    'word1_ortho': w1.get('pWord_ortho', ''),
                    'word2_ortho': w2.get('pWord_ortho', ''),
                    'word1_roman': r1,
                    'word2_roman': r2,
                    'word1_prono': w1.get('pWord_prono', ''),
                    'word2_prono': w2.get('pWord_prono', ''),
                    'insertion_type': ins_type,
                    'actual_n_inserted': actual_n,
                    'filename': fname,
                    'spk_gender': w1.get('spk_gender', ''),
                    'spk_age': w1.get('spk_age', ''),
                    'boundary_type': 'phrase',
                })

    return pd.DataFrame(candidates)

print('구 경계 ㄴ삽입 검색 함수 정의 완료')

구 경계 ㄴ삽입 검색 함수 정의 완료


In [ ]:
# 구 경계 ㄴ삽입 검색 실행
print('구 경계 ㄴ삽입 검색 중...')
df_phrase_n = search_phrase_boundary_n_insertion(df_seoul)
print(f'구 경계 ㄴ삽입 후보: {len(df_phrase_n):,}건')

if len(df_phrase_n) > 0:
    print(f'  실제 ㄴ삽입: {(df_phrase_n["actual_n_inserted"]=="yes").sum():,}건')
    print(f'  ㄴ삽입 없음: {(df_phrase_n["actual_n_inserted"]=="no").sum():,}건')
    print(f'\n상위 10건:')
    print(df_phrase_n[['word1_ortho', 'word2_ortho', 'insertion_type', 'actual_n_inserted']].head(10))

구 경계 ㄴ삽입 검색 중...


## 6. 결과 저장

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 전체 매칭 결과 저장
if len(df_matched) > 0:
    out1 = f'{RESULT_DIR}/seoul_all_phenomena_{timestamp}.csv'
    df_matched.to_csv(out1, index=False, encoding='utf-8-sig')
    print(f'전체 매칭: {out1} ({len(df_matched):,}행)')

# 구 경계 ㄴ삽입 저장
if len(df_phrase_n) > 0:
    df_phrase_n['include'] = ''
    df_phrase_n['realized'] = ''
    out2 = f'{RESULT_DIR}/seoul_phrase_n_insertion_{timestamp}.csv'
    df_phrase_n.to_csv(out2, index=False, encoding='utf-8-sig')
    print(f'구 경계 ㄴ삽입: {out2} ({len(df_phrase_n):,}행)')

print('저장 완료')